# Preparations

### packages

In [ ]:
import os, pathlib, mne, mne_bids, matplotlib, datetime, json, yaml, tkinter
import numpy as np
import pandas as pd
import scipy.fft as fft

Preprocessing Settings

In [126]:
# filtering
l_freq = 0.1 # high-pass filter
h_freq = 40 # low-pass filter

# high- and lowpass filter for ica (according to Winkler et al., 2015)
ica_l_freq = 1
ica_h_freq = 40

# resampling frequency
resample_freq = 250

# power line frequency
line_freq = 50

# Determine reference channel
ref_channel = 'Cz'

# frequency bands for snapshot feature extraction
freq_bands = {"D": [1, 4],
              "T": [4, 8],
              "A": [8, 14],
              "Bl": [14, 23],
              "Bh": [23, 30]}

# Epoch parameters
# snapshot epochs
snapshot_epochs_duration = 2
snapshot_epochs_overlap = 0

# colorchange epochs
colorchange_epochs_tmin = -0.2 
colorchange_epochs_tmax = 1
colorchange_epochs_baseline = (-0.2, 0)


# plotting parameters
plot_scalings = dict(eeg=0.00004,ecg=0.00001)
plot_n_channels = 64
plot_duration = 15
plot_n_epochs = 20

Marker settings and dictionaries

In [127]:
event_dict = {
    "block_start": 4,
    "block_end": 2,
    "trial_end": 64,
    "colorchange": 66,
    "rating_color": 70,
    "rating_time": 128,
    "starfield/nc/vel0/den1": 134,
    "starfield/nc/vel0/den2": 192,
    "starfield/nc/vel1/den1": 196,
    "starfield/nc/vel1/den2": 194,
    "starfield/nc/vel2/den1": 198,
    "starfield/nc/vel2/den2": 32,
    "starfield/cc/vel0/den1": 100,
    "starfield/cc/vel0/den2": 98,
    "starfield/cc/vel1/den1": 102,
    "starfield/cc/vel1/den2": 160,
    "starfield/cc/vel2/den1": 164,
    "starfield/cc/vel2/den2": 162,
    # only used in rest state
    "rs_start": 226,
    "rs_end": 230,
    "rs_rating_start": 16,
    "rs_rating_end" : 20,
}

new_annotation_mapping = {'New Segment/': '99999',
                           'Stimulus/S  2': 'block_end',
                           'Stimulus/S  4': 'block_start',
                           'Stimulus/S  6': "iti_blackscreen",
                           'Stimulus/S 16': 'rs_rating_start',
                           'Stimulus/S 20': 'rs_rating_end',
                           'Stimulus/S 32': 'starfield/nc/vel2/den2',
                           'Stimulus/S 64': 'trial_end',
                           'Stimulus/S 66': 'colorchange',
                           'Stimulus/S 70': 'rating_color',
                           'Stimulus/S 98': 'starfield/cc/vel0/den2',
                           'Stimulus/S100': 'starfield/cc/vel0/den1',
                           'Stimulus/S102': 'starfield/cc/vel1/den1',
                           'Stimulus/S128': 'rating_time',
                           'Stimulus/S134': 'starfield/nc/vel0/den1',
                           'Stimulus/S160': 'starfield/cc/vel1/den2',
                           'Stimulus/S162': 'starfield/cc/vel2/den2',
                           'Stimulus/S164': 'starfield/cc/vel2/den1',
                           'Stimulus/S192': 'starfield/nc/vel0/den2',
                           'Stimulus/S194': 'starfield/nc/vel1/den2',
                           'Stimulus/S196': 'starfield/nc/vel1/den1',
                           'Stimulus/S198': 'starfield/nc/vel2/den1',
                           'Stimulus/S226': 'rs_start',
                           'Stimulus/S230': 'rs_end'}


condition_dict = {
    134: "vel0/den1",
    100: "vel0/den1",
    192: "vel0/den2",
    98: "vel0/den2",
    196: "vel1/den1",
    102: "vel1/den1",
    194: "vel1/den2",
    160: "vel1/den2",
    198: "vel2/den1",
    164: "vel2/den1",
    32: "vel2/den2",
    162: "vel2/den2",
    226: "rs",
    4: "other",
    2: "other",
    6: "other",
    64: "other",
    66: "other",
    70: "other",
    128: "other",
    230: "other",
    16: "other",
    20: "other",
}


# list of all IDs that mark the beginning of a trial
task_ids = [134,192,196,194,198,32,100,98,102,160,164,162]
rs_id = [226]

# all ids for events where a starfield is presented
analysis_ids = task_ids + rs_id


# datatypes for logfile df
datatypes_dict = {'block': 'Int64',
                  'trial': 'Int64',
                  'starting_time': 'Float64',
                  'condition_code': 'Int64',
                  'duration': 'Float64',
                  'velocity': 'Int64',
                  'density': 'Int64',
                  'color_rating': 'object',
                  'color_rating_rt': 'Float64',
                  'time_rating': 'Float64',
                  'time_rating_rt': 'Float64',
                  'color_log': 'object'}


logfile_marker_translation = {
    "11": "starfield/nc/vel0/den1",
    "12": "starfield/nc/vel0/den2",
    "13": "starfield/nc/vel1/den1",
    "14": "starfield/nc/vel1/den2",
    "15": "starfield/nc/vel2/den1",
    "16": "starfield/nc/vel2/den2",
    "21": "starfield/cc/vel0/den1",
    "22": "starfield/cc/vel0/den2",
    "23": "starfield/cc/vel1/den1",
    "24": "starfield/cc/vel1/den2",
    "25": "starfield/cc/vel2/den1",
    "26": "starfield/cc/vel2/den2",
    "30": "rs_start",
    "226": "rs_start"}

# list of all IDs that mark the beginning of a trial
task_ids = [134,192,196,194,198,32,100,98,102,160,164,162]
rs_id = [226]

# all ids for events where a starfield is presented
sf_ids = task_ids + rs_id

channel_name_list_eeg = ['Fp1', 'Fz', 'F3', 'F7', 'FT9', 'FC5', 'FC1', 'C3', 'T7', 'TP9', 'CP5', 'CP1', 'Pz', 'P3', 'P7', 'O1',
                         'Oz', 'O2', 'P4', 'P8', 'TP10', 'CP6', 'CP2', 'C4', 'T8', 'FT10', 'FC6', 'FC2', 'F4', 'F8', 'Fp2', 'AF7',
                         'AF3', 'AFz', 'F1', 'F5', 'FT7', 'FC3', 'C1', 'C5', 'TP7', 'CP3', 'P1', 'P5', 'PO7', 'PO3', 'POz', 'PO4',
                         'PO8', 'P6', 'P2', 'CPz', 'CP4', 'TP8', 'C6', 'C2', 'FC4', 'FT8', 'F6', 'AF8', 'AF4', 'F2', 'FCz', 'Cz']

In [128]:
matplotlib.use('Qt5Agg')

### Directories

In [ ]:
# ENTER PATH TO DATA FOLDER HERE
path_study_data = pathlib.Path("D:/EEGST data/")

# Folder containing sourcedata (behavioral data, demografic data, montages)
path_sourcedata = path_study_data / 'bids_dataset' / 'sourcedata'

# path to preprocessing folder
path_data_preprocessing =  path_study_data / 'preprocessing'
# Create folder if they do not yet exist
if not os.path.isdir(path_data_preprocessing):
    os.mkdir(path_data_preprocessing)

# subject wise parameter folder
path_data_parameter = path_study_data / 'subjectwise_parameter'
# Create folder if they do not yet exist
if not os.path.isdir(path_data_parameter):
    os.mkdir(path_data_parameter)

# Folder group level data
path_data_grouplevel = path_study_data / 'group_level'

# Functions

## Helper Functions

#### *logfile()*

creates a logfile class for initiating and appending to the logfile

In [130]:
class logfile():
    """
    Class for managing a subject-specific preprocessing logfile.

    Args:
        subject (str or int): Subject identifier.
        overwrite (bool): If True, overwrite existing logfile.

    Methods:
        append_line(line): Appends a line to the logfile.
    """
    
    def __init__(self, subject, overwrite = False):
    
        self.subject = subject
        self.path_logfile = path_data_preprocessing_subj / f"preprocessing_logfile_{self.subject}.txt"
        print(f"logfile path: {self.path_logfile}")
        
        # delete old file in case overwrite = True
        if overwrite:
            pathlib.Path(self.path_logfile).unlink(missing_ok=True)
             
        # Create new file and write header line
        if not pathlib.Path(self.path_logfile).is_file():
            self.append_line(f"Preprocessing Logfile - Subject: {self.subject} - Date: {datetime.datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
    
    def append_line(self, line):
        """
        Appends a line to the logfile.

        Args:
            line (str): The line to append.
        """
        
        with open(self.path_logfile, 'a') as f:
            f.writelines([line, '\n'])

#### *update_subject_parameter()*
Updates subject parameter file

In [131]:
class paramfile():
    """
    Class for managing subject-specific parameter files in YAML format.

    Args:
        subject (str or int): Subject identifier.
        load (bool): If True, load parameters from file.

    Methods:
        get(param_name): Retrieve a parameter value.
        update(param_name, param_value): Update or add a parameter.
    """

    # creates file
    def __init__(self, subject, load = True):  
        self.subject = subject
        self.parameters = {}
        self.path_paramfile = path_data_parameter / f"paramfile_{self.subject}.txt"

        if not self.path_paramfile.is_file():
            with open(self.path_paramfile, 'w') as file: 
                pass 
            
    # retrieve parameter
    def get(self, param_name):
        """
        Retrieve a parameter value by name.

        Args:
            param_name (str): Name of the parameter.

        Returns:
            Any: The value of the parameter, or None if not found.
        """
        result = None

        # load parameters from file if it exists
        if self.path_paramfile.is_file():
            with open(self.path_paramfile, 'r') as file:
                self.parameters = yaml.safe_load(file)

        # retrive parameter if it exists
        if self.parameters is not None:
            if param_name in self.parameters:
                result = self.parameters[param_name]
        return(result)

    # update parameter
    def update(self, param_name, param_value):
        """
        Update or add a parameter and save to file.

        Args:
            param_name (str): Name of the parameter.
            param_value (Any): Value to set.
        """

        if self.parameters is None:
            self.parameters = {}
            
        self.parameters[param_name] = param_value
        with open(self.path_paramfile, 'w') as yaml_file:
            yaml.dump(self.parameters, yaml_file, default_flow_style=False)

#### *get_subject_list*
Extracts and return a list of all subject names from demografic data file 

In [ ]:
def get_subject_list(use_raw_demodata = False, exclude_subjects = True, single_group = "controls", subject_selection = []):
    """
    Retrieve a list of subject IDs from demographic data based on filtering criteria.

    Args:
        use_raw_demodata (bool): If True, load raw demographic data. If False, use preprocessed data.
        exclude_subjects (bool): If True, exclude subjects not marked as included.
        single_group (str): Filter to a specific group. Use "controls" for group 1 or "patients" for group 2.
        subject_selection (list[int]): Optional list of subject IDs to include. If provided, only these subjects are returned.

    Returns:
        list[int]: List of subject IDs that meet the specified criteria.
    """
    

    # Load either raw or preprocessed demografic data
    if use_raw_demodata:
        demo_data = pd.read_csv(path_sourcedata / 'demografic_data.csv', encoding='latin-1', sep=";")
    else:
        demo_data = pd.read_csv(pathlib.Path(path_data_grouplevel / 'demografic_availabledata.csv'), encoding='cp1252')

    # only keep subjects from specific group
    if single_group == "controls":
        demo_data = demo_data[demo_data.group == 1]
    elif single_group == "patients":
        demo_data = demo_data[demo_data.group == 2]

    # only keep subjects not marked as excluded
    if exclude_subjects:
        demo_data = demo_data[demo_data.included]

    # extract subject names
    subjects = [int(subject["subject"]) for _ , subject in demo_data.iterrows()]

    if subject_selection:
        subjects = [subject for subject in subjects if subject in subject_selection] 

    return(subjects)

#### *load_show_data()*
loads data file of specified subject and plots it

In [133]:
def load_show_data(subject, file, plot = True):
    """
    Loads and optionally plots EEG data for a given subject and file.

    Args:
        subject (str or int): Subject identifier.
        file (str): Filename to load.
        plot (bool): If True, plot the data.

    Returns:
        mne.io.Raw or mne.Epochs: Loaded EEG data.
    """
    set_current_subject(subject)
    if file[-7:] == 'epo.fif':
        eeg_data = mne.read_epochs(path_data_preprocessing_subj / file)
        if plot:
            eeg_data.plot(title = f"Showing: {subject} / {file}",
                          block = True, scalings = plot_scalings, n_channels = plot_n_channels, n_epochs = plot_n_epochs)

    else:
        eeg_data = mne.io.read_raw_fif(path_data_preprocessing_subj / file)
        if plot:
            eeg_data.plot(title = f"Showing: {subject} / {file}",
                          block = True, scalings = plot_scalings, n_channels = plot_n_channels, duration = plot_duration, clipping = None)
    
    return(eeg_data)

#### *set_current_subject()*
Sets paths to current subject / creates save folder / creates logfile

In [134]:
def set_current_subject(current_subject, new_log = False, print_subject = False):    
    """
    Sets global variables and paths for the current subject, creates folders, and initializes log and parameter files.

    Args:
        current_subject (str or int): Subject identifier.
        new_log (bool): If True, overwrite existing logfile.
    """        

    global subject
    subject = str(current_subject).zfill(3)

    if print_subject:
        print("Current subject:", subject)

    # path to subject wise preprocessing folder
    global path_data_preprocessing_subj
    path_data_preprocessing_subj = path_data_preprocessing / subject
    
    # Create folder if they do not yet exist
    if not os.path.isdir(path_data_preprocessing_subj):
        os.mkdir(path_data_preprocessing_subj)
    
    # subject wise procrocessing logfile
    global log
    log = logfile(subject, overwrite = new_log)
  
    # subject wise processing parameters
    global subj_param
    subj_param = paramfile(subject, load = True)

### *showmessagbox()*

Function to display a message box in front with the options "yes", "no", "cancel" and return the answer

In [135]:
def showmessagebox(questiontext = "Do you want to proceed?"):
    """
    Displays a message box with Yes/No/Cancel options and returns the user's choice.

    Args:
        questiontext (str): The question to display.

    Returns:
        bool or None: True for Yes, False for No, None for Cancel.
    """
    root = tkinter.Tk()

    root.wm_attributes("-topmost", 1)
    root.withdraw()
    result = tkinter.messagebox.askyesnocancel("Question", questiontext, parent = root)

    root.destroy()

    return(result)

## Processing Pipeline

### 1: Preparing data

#### *crop_data()*
cuts before first and after last resting state measurement

#### *add_annotations()*
adds / renames annotations to include keywords and enable pattern matching

#### *prepare dataset()*
adds metadata to dataset, checks triggers, crop_data(), add_annotations(), resamples data, saves eeg and ecg seperately

#### *pipeline_1_auto_downsampling()*

In [ ]:
# Delete intervals that are not interesting (breaks between blocks etc.)
# Identify the first and the last event in events and make sure, it is rs_start (start of resting state) and rs_rating_end (end of resting state)
# raise exception if they do not. Then compute duration of whole measurement and crop afterwards.

def crop_data(uncropped_eeg):
    """
    Crops EEG data before the first and after the last relevant event.

    Args:
        uncropped_eeg (mne.io.Raw): Uncropped EEG data.

    Returns:
        mne.io.Raw: Cropped EEG data.
    """

    # load a copy of raw data for cropping
    uncropped_eeg.load_data()
    
    # get updated list of events
    events_ica, event_id_ica = mne.events_from_annotations(uncropped_eeg, event_id = event_dict)  
    
    # define the duration before the first and after the last event before and after which data should be cropped
    crop_margin_start = 5
    crop_margin_end = 5 

    # extract first event
    first_event = events_ica[0]
    key = list(event_dict.keys())[list(event_dict.values()).index(first_event[2])]
    # make sure that first event is 'rs_start'
    if key == "rs_start":
        # save timepoint first_event in seconds
        timepoint_first_event = (first_event[0] / 1000)
    # if first event is not rs_start
    else:
        if not subj_param.get('ignore_wrong_first_event'):
            subj_param.update('ignore_wrong_first_event', False)
            raise Exception("First event in events list ist not 'rs_start' but: " + key)

    # determine timepoint before of which data will be cropped (making sure it is not < 0)
    crop_before = max(0, (timepoint_first_event - crop_margin_start))


    # extract last event
    last_event = events_ica[-1]
    key = list(event_dict.keys())[list(event_dict.values()).index(last_event[2])]

    
    # make sure that last event is 'rs_rating_end'
    if key == "rs_rating_end" or key == "rs_end":
        # save timepoint last_event in seconds
        timepoint_last_event = (last_event[0] / 1000)
    else:
        print("Last event in list not 'rs_rating_end' or 'rs_end' but:", key)
        if subj_param.get('ignore_wrong_last_event'):
            print("Ignoring wrong last event (see subjectwise parameter)", key)
            # use the last event from event list
            timepoint_last_event = uncropped_eeg.times[-1]
        else:
            # add variable to paramfile
            subj_param.update('ignore_wrong_last_event', False)
            uncropped_eeg.plot()
            raise Exception("Last event in events list ist not 'rs_rating_end' but: " + key)

    # determine timepoint after which data will be cropped (making sure it is > last data point)
    crop_after = min(uncropped_eeg.times[-1], (timepoint_last_event + crop_margin_end))


    # crop whole dataset after time point of last measurement + crop_margin
    cropped_eeg = uncropped_eeg.copy().crop(tmin = crop_before,
                                            tmax = crop_after)

    # print information about how many sec were cropped at the beginning and at the end
    print(round(crop_before, 2), "sec in the beginning and", round((uncropped_eeg.times[-1] - crop_after),2), "sec in the end have been cropped")
    log.append_line(f"\tcropped beginning: {round(crop_before, 2)} / Cropped end: {round((uncropped_eeg.times[-1] - crop_after),2)}")    
       
    return(cropped_eeg)


# Adds annotations e.g. for breaks and rating segments
def add_annotations(eeg_data, id_starts, id_ends, description_suffix, detailed_output = False):
    """
    Adds custom annotations to EEG data between specified event IDs.

    Args:
        eeg_data (mne.io.Raw): EEG data.
        id_starts (list or int): Event IDs marking annotation start.
        id_ends (list or int): Event IDs marking annotation end.
        description_suffix (str): Suffix for annotation description.
        detailed_output (bool): If True, print detailed info.

    Returns:
        mne.io.Raw: EEG data with added annotations.
    """
    
    events, _ = mne.events_from_annotations(eeg_data, event_id = event_dict)
    
    # extract column with event ids and turn it into np-array
    event_id_col = np.array(events[:,2])

    # extract all events
    start_indices = np.where(np.in1d(event_id_col, id_starts))[0]

    # turn np-array into list for index() function
    event_id_col = event_id_col.tolist()

    # create new lists for storing the starts and ends of actual annotations
    annot_starts = []
    annot_ends = []
    # for each start, search the next end-id. Raise exception if none could be found
    for start_index in start_indices: 
        
        try :
            # identifies next element from elements_of_interst in event_id_col    
            if type(id_ends) is list:
                next_end_id = next(filter(lambda x : x in id_ends, event_id_col[start_index:len(event_id_col)]), None) 
            else:
                next_end_id = id_ends
            
            # find index of next end element
            end_index = event_id_col.index(next_end_id, start_index)
            
            annot_starts.append(start_index)
            annot_ends.append(end_index)
            
            if detailed_output:
                print(f"Added annotation of type '{description_suffix}' between {start_index} and {end_index}")
        except ValueError :
            res = f"For event with index {id_starts} no corresponding end [{id_ends}] could be found"
    
    # bind starts and ends
    annot_periods = zip(annot_starts, annot_ends)
    
    onsets = []
    durations = []
    descriptions = []
    for period_nr, period in enumerate(annot_periods):
        onset = events[period[0]][0]
        offset = events[period[1]][0]
        duration = offset - onset
        onsets.append(onset / 1000)
        durations.append(duration / 1000)
        
        if description_suffix == "rs":
            description = f"{description_suffix}_{period_nr+1}"
        else: description = f"{description_suffix}"
        
        descriptions.append(description)
        
    annots = mne.Annotations(
        onset= onsets,  # in seconds    
        duration= durations,  # in seconds, too
        description=descriptions,
        orig_time = eeg_data.info['meas_date']
    )
    
    eeg_data.set_annotations(eeg_data.annotations + annots)  # add to existing
    
    return(eeg_data)



def pipeline_1_auto_preparedataset(subjects, crop = True, add_annots = True, resample = resample_freq, save = True, overwrite = False):
    """
    Prepares raw EEG datasets: adds metadata, checks triggers, crops, annotates, downsamples, and saves.
    Args:
        subjects (list): List of subject identifiers.
        crop (bool): Whether to crop data.
        add_annots (bool): Whether to add annotations.
        resample (int or None): Resampling frequency.
        save (bool): Whether to save the result.
        overwrite (bool): Overwrite existing files.
    """

    for subject in subjects:

        print(f"======================================================================================================================")
        print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Pipeline 1 - Auto - Downsampling // Subject: {subject} ===============") 

        set_current_subject(subject, new_log = True)
        
        # load raw data
        path_bids_data_eeg = mne_bids.BIDSPath(subject=str(subject).zfill(3),
                                               task='Starfield',   # change to colortracking
                                               root=path_study_data / 'bids_dataset',
                                               datatype = 'eeg')
        raw = mne_bids.read_raw_bids(bids_path=path_bids_data_eeg)

        log.append_line(f"Pipeline start - {datetime.datetime.now().strftime('%H:%M:%S')}")
        log.append_line(f"Preparing Dataset - {datetime.datetime.now().strftime('%H:%M:%S')}")

        # --------------------------------
        # Adding infos to dataset
        # specifying power line frequency. (Should be 50 in Europe?)
        raw.info['line_freq'] = line_freq
        
        # load and apply vendor-supplied montage
        montage_fname = path_sourcedata / 'montage' / 'CACS-64_REF.bvef'
        dig_montage = mne.channels.read_custom_montage(montage_fname)
        raw.set_montage(dig_montage)
        
        # Specifying channel types (Change the channel type of the ecg channel to "ecg")
        raw.set_channel_types({'ECG': 'ecg'})
        
    
        # --------------------------------
        # Checking triggers
        events, event_id = mne.events_from_annotations(raw)
        
        global event_dict, new_annotation_mapping
        event_dict_copy = event_dict.copy()
        new_annotation_mapping_copy = new_annotation_mapping.copy()
        
        # find all events that are included in event_dict but not in event_id (trigger that we expect in the data but which are not there)
        not_included = [x for x in event_dict_copy.values() if x not in event_id.values()]

    
        # for each missing event key check whether it is important or just a ct-trial. In the latter case, delete the key from event_dict
        for element in not_included:
            # get the key from the value
            missing_element = list(event_dict.keys())[list(event_dict.values()).index(element)]

            if "cc" in missing_element or missing_element == "rs_rating_start" or missing_element == "rs_rating_end":
                log.append_line(f"\tMissing marker: {missing_element}")
                print (f"Missing marker: {missing_element}")
                
                event_dict_copy.pop(missing_element)
                # delete event from new_annotations_mapping
                new_annotation_mapping_copy = {key:val for key, val in new_annotation_mapping_copy.items() if val != missing_element}
            else:
                if not subj_param.get(f"ignore_missing_event_{missing_element}"):
                    # add variable to paramfile
                    subj_param.update(f"ignore_missing_event_{missing_element}", False)
                    raise Exception(f"Event: '{missing_element}' is missing in data")

        
        # add fixed annotation ids
        raw.annotations.rename(new_annotation_mapping_copy)
        print(f"new_annotation_mapping_copy: {new_annotation_mapping_copy}")
        
        
        # Cropping data before first and after last marker
        if crop:
            raw = crop_data(raw)     
        
        
        if add_annots:
            log.append_line(f"\tAdding annotations: rs, starfield, ratings, breaks")
            # rs
            # adding annotations for resting state
            raw = add_annotations(raw, 226, 230, "rs")
            # adding annotations for ratings after resting state
            raw=add_annotations(raw, 230, 20, "bad_rating_rs")
            
            # blocks
            # adding annotation for break after 1. resting state
            raw = add_annotations(raw, 20, task_ids, "bad_breaks")
            # adding annotations for breaks after blocks
            raw = add_annotations(raw, 2, task_ids + [226], "bad_breaks")
            
            # trials
            # adding annotations for trial stimuli
            raw = add_annotations(raw, task_ids, 70, "starfield")
            # adding annotations for trial ratings
            raw = add_annotations(raw, 70, 64, "bad_rating_trial")
            # adding annotations for inter-trial-intervals
            raw = add_annotations(raw, 64, task_ids + [2], "bad_iti")
            
            
            # before first and after second resting states
            # extract first event 
            events, event_id = mne.events_from_annotations(raw, event_id = event_dict_copy)
            timepoint_first_event = (events[0][0] / 1000)
            onsets = [timepoint_first_event - 5]
            durations = [min(timepoint_first_event, 5)]
            
            # extract last event
            last_event = events[-1]
            key = list(event_dict.keys())[list(event_dict_copy.values()).index(last_event[2])]
            # make sure that last event is 'rs_rating_end'
            if key == "rs_rating_end":
                # save timepoint last_event in seconds
                timepoint_last_event = (last_event[0] / 1000)
                onsets.append(timepoint_last_event)
                durations.append(5)
            
            # create annotations
            annots = mne.Annotations(
                onset= onsets,  # in seconds    
                duration= durations,  # in seconds, too
                description="bad_breaks",
                orig_time = raw.info['meas_date']
                )
            
            # add annotations
            for annot in annots:
                raw.annotations.append(annot['onset'], annot['duration'], annot['description'])

            
        if resample:
            log.append_line(f"\tDownsampling to {resample}Hz.")
            
            events, event_id = mne.events_from_annotations(raw)
        
            # downsampling
            resampled_eeg, _ = raw.copy().resample(resample, events = events)

            raw = resampled_eeg.copy()
        
        # save dataset
        if save:
            raw.save(path_data_preprocessing_subj / '1_raw_eeg.fif', overwrite=overwrite)

### 2: Automatic low pass filtering

#### *pipeline_2_auto_lowpassfiltering()*

In [137]:
def pipeline_2_auto_lowpassfiltering(subjects, save = True, overwrite = False):
    """
    Applies low-pass filtering to EEG data for each subject.

    Args:
        subjects (list): List of subject identifiers.
        save (bool): Whether to save the result.
        overwrite (bool): Overwrite existing files.
    """

    for subject in subjects:

        print(f"======================================================================================================================")
        print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Pipeline 2 - Auto - lowpass filtering // Subject: {subject} ===============") 

        set_current_subject(subject)
        
        log.append_line(f"Auto lowpass filtering - {datetime.datetime.now().strftime('%H:%M:%S')}")
        log.append_line(f"\tFilter: lowpass: {h_freq} / fir_window: hamming")
        log.append_line(f"\tICA Filter: lowpass: {ica_h_freq} / fir_window: hamming")
        
        # load and copy data
        lowpass_raw = mne.io.read_raw_fif(path_data_preprocessing_subj / '1_raw_eeg.fif', preload = True)
    
        # filter the data for ica (High-pass with 1. Hz cut-off is recommended for ICA)
        lowpass_raw.filter(l_freq = None, h_freq = h_freq, fir_window = "hamming")

        # save data
        if save:
            lowpass_raw.save(pathlib.Path(path_data_preprocessing_subj) / '2_lp_eeg.fif', overwrite=overwrite)

### 3: Manual exclusion of bad channels

#### *pipeline_3_manual_badchannelexclusion()*
Plots data for visual checking and manual exclusion of bad (e.g. flat, noisy) channels

In [138]:
def pipeline_3_manual_badchannelexclusion(subjects, use_old_bads = True, save = True, overwrite = False):
    """
    Allows manual or automatic exclusion of bad EEG channels for each subject.

    Args:
        subjects (list): List of subject identifiers.
        use_old_bads (bool): Use previously identified bad channels.
        save (bool): Whether to save the result.
        overwrite (bool): Overwrite existing files.
    """

    for subject in subjects:

        print(f"======================================================================================================================")
        print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Pipeline 3 - Visual bad channel detection // Subject: {subject} ===============")

        set_current_subject(subject)

        log.append_line(f"Bad Channel detection - {datetime.datetime.now().strftime('%H:%M:%S')}")
        if use_old_bads:
            log.append_line(f"\tBad channel info from previous analysis")  
        else:
            log.append_line(f"Bad channel identification by visual inspection")
        
        # Load data
        badchannel_raw = mne.io.read_raw_fif(path_data_preprocessing_subj / '2_lp_eeg.fif')

        if use_old_bads:
            # Load data from previous run
            old_badchannel_raw = mne.io.read_raw_fif(path_data_preprocessing_subj / "3_lp_nobad_eeg.fif")
            # Copy information about bad channels to new data set
            badchannel_raw.info['bads'] = old_badchannel_raw.info['bads']               
        else:
            # Create copy of data and plot it for visual inspection and identification of bad channels
            badchannel_raw.plot(title = f"{subject} - Visual Inspection 1: Check for bad channels", block = True,
                            scalings = plot_scalings, n_channels = plot_n_channels, duration = plot_duration, clipping = None)

        print(f"channels marked as bad {badchannel_raw.info['bads']}")
        log.append_line(f"\tchannels marked as bad: {badchannel_raw.info['bads']}")

        # save data
        if save:
            badchannel_raw.save(path_data_preprocessing_subj / "3_lp_nobad_eeg.fif", overwrite=overwrite)

### 4: Automatic high pass filtering

#### *pipeline_4_auto_highpassfiltering()*

In [139]:
def pipeline_4_auto_highpassfiltering(subjects, save = True, overwrite = False):
    """
    Applies high-pass filtering to EEG data and creates a parallel ICA dataset.

    Args:
        subjects (list): List of subject identifiers.
        save (bool): Whether to save the result.
        overwrite (bool): Overwrite existing files.
    """

    for subject in subjects:

        print(f"======================================================================================================================")
        print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Pipeline 4 - Auto - highpass filtering // Subject: {subject} ===============") 

        set_current_subject(subject)
        
        log.append_line(f"Auto highpass filtering - {datetime.datetime.now().strftime('%H:%M:%S')}")
        log.append_line(f"\tFilter: highpass:{l_freq} / fir_window: hamming")
        log.append_line(f"\tICA Filter: highpass:{ica_l_freq} / fir_window: hamming")

        # load data
        highpass_raw = mne.io.read_raw_fif(path_data_preprocessing_subj / '3_lp_nobad_eeg.fif', preload = True)

        # Create parallel dataset for ica
        ica_parallel_dataset = highpass_raw.copy()
 
        # filter the main data (High-pass with 0.1 Hz cut-off)
        highpass_raw.filter(l_freq = l_freq, h_freq = None, fir_window = "hamming")

        # filter the parallel dataset for ica (High-pass with 1. Hz cut-off for ICA)
        ica_parallel_dataset.filter(l_freq = ica_l_freq, h_freq = None, fir_window = "hamming")

        # save data
        if save:
            highpass_raw.save(pathlib.Path(path_data_preprocessing_subj) / '4a_lp_nobad_hp_eeg.fif', overwrite=overwrite)
            ica_parallel_dataset.save(pathlib.Path(path_data_preprocessing_subj) / '4b_lp_nobad_hpica_eeg.fif', overwrite=overwrite)

### 5: Manual exclusion of bad segments in parallel data sets

#### *pipeline_5_manual_badsegmentexclusion()*

In [140]:
def pipeline_5_manual_badsegmentexclusion(subjects, use_old_annotations = True, save = True, overwrite = False):
    """
    Allows manual or automatic exclusion of bad segments in ICA data for each subject.

    Args:
        subjects (list): List of subject identifiers.
        use_old_annotations (bool): Use previously identified bad segments.
        save (bool): Whether to save the result.
        overwrite (bool): Overwrite existing files.
    """

    for subject in subjects:

        print(f"======================================================================================================================")
        print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Pipeline 5 - Manual - Cleaning ICA data // Subject: {subject} ===============")

        set_current_subject(subject)   

        log.append_line(f"Manual cleaning ICA data - {datetime.datetime.now().strftime('%H:%M:%S')}")
        if use_old_annotations:
            log.append_line(f"\tBad segment annotations for ICA from previous run")
        else:
            log.append_line(f"Bad segment annotations for ICA by visual inspection")
            

        # Load and create copy of data
        input_data = mne.io.read_raw_fif(path_data_preprocessing_subj / '4b_lp_nobad_hpica_eeg.fif')

        annotated_data = input_data.copy()

        if use_old_annotations:
            # Load annotations from old data set and copy it to new data set
            old_data = mne.io.read_raw_fif(path_data_preprocessing_subj / '4b_lp_nobad_hpica_eeg.fif')
            annotated_data.set_annotations(old_data.annotations)
        else:
            # plot data for manual marking of annotations
            annotated_data.plot(title = f"{subject} - Visual Inspection: Bad segment exclusion ica data", block = True,
                                scalings = plot_scalings, n_channels = plot_n_channels, duration = plot_duration, clipping = None)

        # Log info about number of annotations
        log.append_line(f"\tadded annotations: {len(annotated_data.annotations)}")

        # save data
        if save:
            annotated_data.save(path_data_preprocessing_subj / "5_lp_nobad_hpica_clean_eeg.fif", overwrite=overwrite)

### 6: Automatically fitting ICA

#### *auto_fitica()*
#### *pipeline_6_auto_fittingica()*

In [141]:
def pipeline_6_auto_fittingica(subjects, use_old_ica = True, save = True, overwrite = False):
    """
    Fits ICA to cleaned EEG data, detects EOG components, and saves ICA results.

    Args:
        subjects (list): List of subject identifiers.
        use_old_ica (bool): Use previously computed ICA.
        save (bool): Whether to save the result.
        overwrite (bool): Overwrite existing files.
    """

    for subject in subjects:

        print(f"======================================================================================================================")
        print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Pipeline 6 - Auto - Fitting ICA // Subject: {subject} ===============")

        set_current_subject(subject)

        clean_ica_parallel_dataset = mne.io.read_raw_fif(path_data_preprocessing_subj / '5_lp_nobad_hpica_clean_eeg.fif')

        log.append_line(f"Auto fitting ICA - {datetime.datetime.now().strftime('%H:%M:%S')}")
        if use_old_ica:
            log.append_line(f"\tICA from previous run")
        else:
            log.append_line(f"New ICA")


        if not use_old_ica:
            
            # ICA parameters
            fit_params = dict(fastica_it=5) # Number of fast ica iterations before starting the proper algorithm
            random_state = 42 # initialize the random generator with the same starting parameter, resulting in the same results every time the script is run

            ica = mne.preprocessing.ICA(n_components=999,
                                        method='picard',
                                        max_iter=100,
                                        fit_params=fit_params,
                                        random_state=random_state)

            # fitting ica
            ica.fit(clean_ica_parallel_dataset)

            # ------------------------------------------
            # detecting eog patterns

            # identify EOG component (based on channels Fp1) 
            eog_epochs = mne.preprocessing.create_eog_epochs(clean_ica_parallel_dataset,
                                                            ch_name="Fp1",
                                                            reject=None,
                                                            baseline=(None, -0.2),
                                                            tmin=-0.5, tmax=0.5)

            eog_inds, eog_scores = ica.find_bads_eog(inst=eog_epochs,
                                                    ch_name=["Fp1", "F8"],
                                                    threshold=1,
                                                    reject_by_annotation = False)

            # tells object of ica class which components (indices) should be excluded
            ica.exclude = eog_inds
        
        else:
            # load old ica
            ica = mne.preprocessing.read_ica(pathlib.Path(path_data_preprocessing_subj) / '6a_auto_ica.fif')

            # load eog scores
            eog_scores = np.load((pathlib.Path(path_data_preprocessing_subj) / '6b_eog_scores.npy'), allow_pickle=True)
            eog_scores = eog_scores.tolist()

            # load eog_epochs
            eog_epochs = mne.read_epochs(path_data_preprocessing_subj / '6c_eog_epo.fif')

            
        print(f"ICA components suggested for exclusion: {ica.exclude}")
        log.append_line(f"\tICA components suggested for exclusion: {ica.exclude}")

        if save:
            # save ICA
            ica.save(pathlib.Path(path_data_preprocessing_subj) / '6a_auto_ica.fif', overwrite=overwrite)

            # save eog_scores as numpy              
            np.save((pathlib.Path(path_data_preprocessing_subj) / '6b_eog_scores'), np.array(eog_scores, dtype=object), allow_pickle=True)

            # save eog_epochs
            eog_epochs.save(path_data_preprocessing_subj / '6c_eog_epo.fif', overwrite=overwrite)

### 7: Manual identification of EOG components

#### *manual_identifyicacomponents()*
#### *pipeline_7_manual_identifyingicacomponents()*

In [142]:
def pipeline_7_manual_identifyingicacomponents(subjects, use_old_ica_components = True, save = True, overwrite = False):
    """
    Allows manual identification of ICA components related to EOG for each subject.

    Args:
        subjects (list): List of subject identifiers.
        use_old_ica_components (bool): Use previously identified ICA components.
        save (bool): Whether to save the result.
        overwrite (bool): Overwrite existing files.
    """

    for subject in subjects:

        print(f"======================================================================================================================")
        print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Pipeline 7 - Manual - Choosing EOG related ICA components // Subject: {subject} ===============")

        set_current_subject(subject)

        log.append_line(f"Manual inspection ICA components - {datetime.datetime.now().strftime('%H:%M:%S')}")
        if use_old_annotations:
            log.append_line(f"\tICA components from previous run")
        else:
            log.append_line(f"New ICA components by visual inspection")

        # load auto ica
        ica = mne.preprocessing.read_ica(pathlib.Path(path_data_preprocessing_subj) / '6a_auto_ica.fif')


        if not use_old_ica_components:
            ica_parallel_dataset = mne.io.read_raw_fif(path_data_preprocessing_subj / '5_lp_nobad_hpica_clean_eeg.fif')

            # load eog_scores
            eog_scores = np.load((pathlib.Path(path_data_preprocessing_subj) / '6b_eog_scores.npy'), allow_pickle=True)
            eog_scores = eog_scores.tolist()
        
            # load eog_epochs
            eog_epochs = mne.read_epochs(path_data_preprocessing_subj / '6c_eog_epo.fif')

            # Show the components identified by the ICA in an interactive plot and let user choose the relevant components
            ica.plot_components(inst=ica_parallel_dataset, nrows = 5)

            # Indicator of the probability of each component that it is related to eog (?)
            ica.plot_scores(eog_scores)

            # --> For blinks it should be around 0 (-200 - 200)
            ica.plot_overlay(eog_epochs.average())
        else:
            # load old ica file containing info about excluded components
            old_ica = mne.preprocessing.read_ica(pathlib.Path(path_data_preprocessing_subj) / '7_manual_ica.fif')
            # copy excluded components to new ica
            ica.exclude = old_ica.exclude
                    

        if save:
            # saving ica object
            ica.save(pathlib.Path(path_data_preprocessing_subj) / '7_manual_ica.fif', overwrite=overwrite)

### 8: Automatic applying ICA

#### *auto_applyica()*
#### *pipeline_8_auto_apllyica()*

In [143]:
def pipeline_8_auto_apllyica(subjects, save = True, overwrite = False):
    """
    Applies ICA to main EEG data and checks event alignment before and after ICA.

    Args:
        subjects (list): List of subject identifiers.
        save (bool): Whether to save the result.
        overwrite (bool): Overwrite existing files.
    """

    for subject in subjects:

        print(f"======================================================================================================================")
        print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Pipeline 8 - Auto - Applying ICA // Subject: {subject} ===============")

        set_current_subject(subject)

        log.append_line(f"Auto applying ICA - {datetime.datetime.now().strftime('%H:%M:%S')}")

        # load ica
        ica = mne.preprocessing.read_ica(pathlib.Path(path_data_preprocessing_subj / '7_manual_ica.fif'))

        # Load raw data
        main_data = mne.io.read_raw_fif(path_data_preprocessing_subj / '4a_lp_nobad_hp_eeg.fif', preload = True)

        # ------------------------------------------
        # applying ica
        ica_eeg = ica.apply(main_data.copy())

        # tells object of ica class which components (indices) should be excluded
        excluded_components = ica.exclude 

        print("Excluded ICA components:", excluded_components)
        log.append_line(f"\texlcuded ICA components: {excluded_components}")

        # ------------------------------------------
        # make sure that event timepoints did not change by ica by comparing events before and after ica
        # get new events after ICA
        events, _ = mne.events_from_annotations(main_data, event_id = event_dict)
        events_new, _ = mne.events_from_annotations(ica_eeg, event_id = event_dict)
        # compare for each event
        event_results = events == events_new
        # check wheter all comparissons resulted in 'True' and raise exception if not
        if not event_results.all():
            raise Exception("Events before and afte ICA do not align")

        # saving data
        if save:
            ica_eeg.save(pathlib.Path(path_data_preprocessing_subj) /'8_lp_nobad_hp_ica_eeg.fif', overwrite=overwrite)

### 9: Manual comparing before after ICA

#### *pipeline_9_manual_comparebeforeafterica()*

This function does not produce or save any new data but is merely for visual inspection

In [144]:
def pipeline_9_manual_comparebeforeafterica(subjects, run_without_plotting = False):
    """
    Plots and compares EEG data before and after ICA for visual inspection.

    Args:
        subjects (list): List of subject identifiers.
        run_without_plotting (bool): If True, skip plotting.
    """

    for subject in subjects:

        print(f"======================================================================================================================")
        print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Pipeline 9 - Manual - Compare before after ICA // Subject: {subject} ===============")

        set_current_subject(subject)

        log.append_line(f"Manual comparrison before/after - {datetime.datetime.now().strftime('%H:%M:%S')}")

        if not run_without_plotting:
            data_before = mne.io.read_raw_fif(path_data_preprocessing_subj / '4a_lp_nobad_hp_eeg.fif', preload = True)
            data_after = mne.io.read_raw_fif(path_data_preprocessing_subj /'8_lp_nobad_hp_ica_eeg.fif', preload = True)
            
            raw_selection_before = data_before.get_data()
            raw_selection_after = data_after.get_data()
            
            raw_diff_data = np.subtract(raw_selection_after, raw_selection_before)
            raw_diff = mne.io.RawArray(raw_diff_data, data_before.info)

            data_before.plot(title = f'{subject} - Before ICA',
                            scalings = dict(eeg=0.00002,ecg=0.00001), n_channels = plot_n_channels, duration = plot_duration, clipping = None)

            data_after.plot(title = f'{subject} - After ICA',
                            scalings = dict(eeg=0.00002,ecg=0.00001), n_channels = plot_n_channels, duration = plot_duration, clipping = None)
            
            raw_diff.plot(title = f'{subject} - Diff before-after ICA',
                        scalings = dict(eeg=0.00002,ecg=0.00001), n_channels = plot_n_channels, duration = plot_duration, clipping = None,
                        block = True)

### 10: Automatic interpolating bad channels & rereferencing

#### *rereference()*

function that re-references data to given electrode
#### *pipeline_10_auto_interpolateandrereference()*

In [ ]:
def pipeline_10_auto_interpolateandrereference(subjects, save = True, overwrite = False):
    """
    Interpolates bad channels and re-references EEG data to a specified reference.

    Args:
        subjects (list): List of subject identifiers.
        save (bool): Whether to save the result.
        overwrite (bool): Overwrite existing files.
    """

    for subject in subjects:

        print(f"======================================================================================================================")
        print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Pipeline 10 - Auto - Interpolating bad channels & rereferencing // Subject: {subject} ===============")
    
        set_current_subject(subject)

        log.append_line(f"Auto interpolation and rereferencing - {datetime.datetime.now().strftime('%H:%M:%S')}")

        # load data
        input_data = mne.io.read_raw_fif(path_data_preprocessing_subj /'8_lp_nobad_hp_ica_eeg.fif', preload = True)
        
        # interpolate bad channels 
        interpolated_data = input_data.interpolate_bads(reset_bads = True)
        log.append_line(f"\tinterpolated channels: {interpolated_data.info['bads']}")

        # add a new zero channel to the data
        reref_data = interpolated_data.add_reference_channels(ref_channel)
        # change the reference to a common average reference (by subtracting the current reference channel from all other channels (?))
        reref_data.set_eeg_reference(ref_channels='average')
        
        # load and apply vendor-supplied montage for average reference
        montage_fname = path_sourcedata / 'montage' / 'CACS-64_NO_REF.bvef'
        dig_montage = mne.channels.read_custom_montage(montage_fname)
        montage_data = reref_data.set_montage(dig_montage)
        
        # saving data
        if save:
            montage_data.save(pathlib.Path(path_data_preprocessing_subj) / '10_lp_nobad_hp_ica_reref_eeg.fif', overwrite=overwrite)

### 11: Manual exclusion of artefacts in main data

#### *pipeline_11_manual_excludingartefacts()*

In [146]:
def pipeline_11_manual_excludingartefacts(subjects, show_data = True, use_old_annotations = True, use_ica_annotations = False, save = True, overwrite = False):
    """
    Allows manual or automatic exclusion of artefacts in main EEG data.

    Args:
        subjects (list): List of subject identifiers.
        show_data (bool): If True, show data for manual annotation.
        use_old_annotations (bool): Use previous artefact annotations.
        use_ica_annotations (bool): Use ICA artefact annotations.
        save (bool): Whether to save the result.
        overwrite (bool): Overwrite existing files.
    """

    for subject in subjects:

        print(f"======================================================================================================================")
        print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Pipeline 11 - Manual - Excluding artefacts // Subject: {subject} ===============")

        set_current_subject(subject)

        log.append_line(f"Manual exclusion artefacts - {datetime.datetime.now().strftime('%H:%M:%S')}")
        if use_old_annotations:
            log.append_line(f"\tBad segment annotations from previous run")
        else:
            log.append_line(f"Bad segment annotations by visual inspection")

            
        # Load data
        main_dataset = mne.io.read_raw_fif(path_data_preprocessing_subj / '10_lp_nobad_hp_ica_reref_eeg.fif')
        # store list of original annotations for later comparisson
        original_annotations = main_dataset.annotations


        # Add annotations from previous run
        if use_old_annotations:

            old_dataset = mne.io.read_raw_fif(path_data_preprocessing_subj / '11a_lp_nobad_hp_ica_reref_clean_eeg.fif')

            # extract all new annotations from the cleaned data for ICA       
            filtered_old_annotations = [annot for annot in old_dataset.annotations if annot not in main_dataset.annotations]

            # add new annotations to main data
            for annot in filtered_old_annotations:
                main_dataset.annotations.append(annot['onset'], annot['duration'], annot['description'])


        # Add annotations from ICA
        if use_ica_annotations: 
            # load ica data
            ica_dataset = mne.io.read_raw_fif(path_data_preprocessing_subj / '5_lp_nobad_hpica_clean_eeg.fif')

            # extract all new annotations from the cleaned data for ICA       
            filtered_ica_annotations = [annot for annot in ica_dataset.annotations if annot not in main_dataset.annotations]

            # add new annotations to main data
            for annot in filtered_ica_annotations:
                main_dataset.annotations.append(annot['onset'], annot['duration'], annot['description'])


        if show_data:
            # start a loop for adding annotations (loop allows to continue manual annotation marking in case plotting window was accidentally closed)
            proceed = True
            while proceed:

                # Create copy of data for manual annotations
                manual_annot_data = main_dataset.copy()

                # plot data and block for manual annotations
                manual_annot_data.plot(title = f"{subject} - Visual Inspection 3: Bad segment exclusion main data", block = True,
                                        scalings = plot_scalings, n_channels = plot_n_channels, duration = plot_duration, clipping = None)
                
                # ask whether results should be saved
                answer = showmessagebox("save results before ending manual annotation marking? (close window to proceed manual annotation marking)")

                if answer == True: # quit and save data
                    main_dataset.set_annotations(manual_annot_data.annotations)
                    proceed = False
                elif answer == False: # quit without saving data
                    proceed = False
                # proceed if answer == none


        if save: 
            # saving eeg data
            main_dataset.save(pathlib.Path(path_data_preprocessing_subj) / '11a_lp_nobad_hp_ica_reref_clean_eeg.fif', overwrite=overwrite)

            # turning annotations object into string for saving as json
            json_data = json.dumps(main_dataset.annotations, indent=4, sort_keys=True, default=str)

            # save annotation data
            with open(path_data_preprocessing_subj / "11b_artefact_annotations.json", "w") as outfile:
                outfile.write(json_data)

### 12: Auto epoching

#### *create_snapshot_epochs()*
#### *create_colorchange_epochs()*
#### *pipeline_12_auto_epoching()*

Building epochs out of events

In [ ]:
def create_snapshot_epochs(eeg_orig, plotting = False,
                           duration = snapshot_epochs_duration, overlap = snapshot_epochs_overlap,
                           save = True, overwrite = False):
    """
    Creates snapshot epochs from continuous EEG data, splitting trials into fixed-duration segments.

    Args:
        eeg_orig (mne.io.Raw): Original EEG data.
        plotting (bool): If True, plot the epochs.
        duration (float): Duration of each snapshot (seconds).
        overlap (float): Overlap between snapshots (seconds).
        save (bool): Whether to save the epochs.
        overwrite (bool): Overwrite existing files.

    Returns:
        mne.Epochs: Snapshot epochs.
    """  
    
    log.append_line(f"\tsnapshot duration: 2000")

    # extract events
    events, event_id = mne.events_from_annotations(eeg_orig, event_id = event_dict)

    # ------------------------------------------
    # Add events for individual snappshots

    # Currently, each 20 sec interval of starfield color tracking task is signaled by one marker in the beginning.
    # Instead, we want to split the interval in several (potentially overlapping) snapshots of freely adjustable duration and overlap.


    # Step sizes between snapshots and duration of snapshots

    snapshot_steps = round(2000 / (1000 / eeg_orig.info['sfreq']))
    snapshot_duration = round(2000 / (1000 / eeg_orig.info['sfreq']))
        
    events_extended = []
    for event_nr, event in enumerate(events):
        # if event is a stimulus, create and insert several snapshots instead of a single event
        if not event[2] == 66:

            # start of the next event in list
            if (event_nr+1) < len(events):
                next_event_start = events[event_nr+1][0]
            else:
                next_event_start = int((eeg_orig.times[-1] * eeg_orig.info['sfreq']) + eeg_orig.first_samp)
            
            # compute duration of trial
            distance_next_event = next_event_start - event[0]            
            
            # compute number of snapshots to be inserted
            snapshots_total = distance_next_event // snapshot_steps
            
            # create new list of events
            snapshot_list = []
            
            for snapshot in range(0, snapshots_total):
                # start of the snapshot
                snapshort_start = event[0] + (snapshot * snapshot_steps)
                
                # if another snapshot fits into this event, append the snapshot to snapshots_list
                if snapshort_start + snapshot_duration <= next_event_start:
                    snapshot_list.append(np.asarray([event[0] + (snapshot * snapshot_steps), event[1], event[2]]))
            
            # append snapshot_list to new (extended) event list
            events_extended += snapshot_list
        # if event is not a stimulus, simply append to list of events  
        else:
            events_extended.append(event)
        
    events_extended = np.array(events_extended)

    # Create copy of annotations with durations of all bad annotations reduced by .1 sec (where possible)
    onsets = []
    durations = []
    descriptions = []

    for annot_nr, annot in enumerate(eeg_orig.annotations):

        duration = annot["duration"].copy()

        # reduce duration ob "bad"-annotations by .25 so that they do not overlap with the interval used as baseline for subsequent epochs
        if "bad" in annot["description"] and duration >= .25:
            duration -= .25

        # deduct duration between start and first sample divided by sampling frequency from onset
        onsets.append(annot["onset"] - (eeg_orig.first_samp / eeg_orig.info['sfreq']))
        durations.append(duration)
        descriptions.append(annot["description"])

    #onsets =- (eeg_orig.first_samp / eeg_orig.info['sfreq'])
        
    # create new annotation object
    new_annotations = mne.Annotations( 
        onset = onsets,
        duration = durations,
        description = descriptions,
    )  

    # create copy of eeg data with shorter annotations
    eeg_shorter_annotations = eeg_orig.copy()
    eeg_shorter_annotations.set_annotations(new_annotations)


    # Epoching
    snapshot_epochs = mne.Epochs(raw = eeg_shorter_annotations,
                                 events = events_extended,
                                 event_id = event_dict,
                                 tmin = -0.2,
                                 tmax = 2,
                                 baseline = (-.2, 0),
                                 on_missing='warn')


    #load and apply vendor-supplied montage
    montage_fname = path_sourcedata / 'montage' / 'CACS-64_NO_REF.bvef'
    dig_montage = mne.channels.read_custom_montage(montage_fname)
    snapshot_epochs.set_montage(dig_montage)

    if plotting:
        snapshot_epochs.plot(scalings = plot_scalings, n_channels = plot_n_channels, duration = plot_duration, clipping = None)
    
    if save:
        snapshot_epochs.save(path_data_preprocessing_subj / '12a_snapshot_epo.fif', overwrite=overwrite)
        
    return(snapshot_epochs)



def create_colorchange_epochs(eeg_orig, plotting = False,
                              tmin = colorchange_epochs_tmin, tmax = colorchange_epochs_tmax, baseline = colorchange_epochs_baseline,
                              save = True, overwrite = False):
    """
    Creates epochs around color change events in the EEG data.

    Args:
        eeg_orig (mne.io.Raw): Original EEG data.
        plotting (bool): If True, plot the epochs.
        tmin (float): Start time before event (seconds).
        tmax (float): End time after event (seconds).
        baseline (tuple): Baseline interval.
        save (bool): Whether to save the epochs.
        overwrite (bool): Overwrite existing files.

    Returns:
        mne.Epochs: Color change epochs.
    """
    
    log.append_line(f"\tcolorchange epochs tmin: {tmin} / colorchange epochs tmax: {tmax} / colorchange epochs baseline: {baseline}")

    # get original events
    original_events, original_event_id = mne.events_from_annotations(eeg_orig, event_id = event_dict)

    colorchange_epochs = mne.Epochs(raw = eeg_orig,
                                    events = original_events,
                                    event_id = dict(colorchange = 66),
                                    tmax = tmax,
                                    baseline = baseline)

    # load and apply vendor-supplied montage
    montage_fname = path_sourcedata / 'montage' / 'CACS-64_NO_REF.bvef'
    dig_montage = mne.channels.read_custom_montage(montage_fname)
    colorchange_epochs.set_montage(dig_montage)

    if plotting:
        colorchange_epochs.plot(scalings = plot_scalings, n_channels = plot_n_channels, duration = plot_duration, clipping = None)
    
    if save:
        colorchange_epochs.save(path_data_preprocessing_subj / '12b_colorchange_epo.fif', overwrite=overwrite)
        
    return(colorchange_epochs)




def pipeline_12_auto_epoching(subjects, save = True, overwrite = False):
    """
    Runs automatic epoching for each subject, creating snapshot and color change epochs.

    Args:
        start (int): Starting subject index.
        end (int or None): Ending subject index.
        save (bool): Whether to save the result.
        overwrite (bool): Overwrite existing files.
    """

    for subject in subjects:

        print(f"======================================================================================================================")
        print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Pipeline 12 - Auto - Epoching // Subject: {subject} ===============")
    
        set_current_subject(subject)

        log.append_line(f"Auto epoching - {datetime.datetime.now().strftime('%H:%M:%S')}")

        # load data
        clean_eeg = mne.io.read_raw_fif(path_data_preprocessing_subj / '11a_lp_nobad_hp_ica_reref_clean_eeg.fif', preload = True)
        
        # epoching
        # create epochs for each snapshot
        create_snapshot_epochs(clean_eeg, save = save, overwrite = overwrite)

        # create epochs for each colorchange
        create_colorchange_epochs(clean_eeg, save = save, overwrite = overwrite)



### 13: Feature extraction

#### *extract_epoch_features()*

Function that extracts features (5 frequency bands per channel (64) = 320 features)

#### *pipeline_13_auto_featureextraction()*


In [148]:
def pipeline_13_auto_featureextraction(subjects, save = True):
    """
    Extracts frequency band features from snapshot epochs for each subject.

    Args:
        subjects (list): List of subject identifiers.
        save (bool): Whether to save the result.
    """

    for subject in subjects:

        print(f"======================================================================================================================")
        print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Pipeline 13 - Auto - Feature Extraction // Subject: {subject} ===============")

        set_current_subject(subject)

        log.append_line(f"Extracting features - {datetime.datetime.now().strftime('%H:%M:%S')}")

        # load data
        epochs = mne.read_epochs(path_data_preprocessing_subj / '12a_snapshot_epo.fif', preload = True)

        # make sure that no channel is marked as bad anymore (they were already interpolated in an earlier step)
        epochs.info['bads'] = []

        # Create dictionary
        feature_dict = {}
        for channel in channel_name_list_eeg:
            for key in freq_bands.keys():
                feature_name = f"{channel}_{key}"
                feature_dict[feature_name] = []

        # crop epochs so that data do not include baseline interval
        epochs_filtered = epochs.copy().crop(0, 2)
        epoch_data = epochs_filtered.pick("eeg").get_data()

        # Calculate the number of points for the FFT with 0.5 Hz resolution
        sfreq = epochs_filtered.info['sfreq']
        n_fft_points = int(sfreq / 0.5)  # Number of points for FFT to achieve 0.5 Hz resolution

        # Perform FFT
        fft_data = fft.rfft(epoch_data, n=n_fft_points)
        freqs = fft.rfftfreq(n_fft_points, d=1/sfreq)

        # turn complex to real numbers
        amplitudes = np.abs(fft_data)

        # Note to self:
        # Explanatin of PSD Calculation from https://vibrationresearch.com/blog/fft-psd-difference/:
        # The process starts with a random time waveform. The algorithm takes frames of equal length and applies the FFT to each frame. Then, it calculates the squared magnitude for each frequency bin and finds the average. The mean-square value is the “power” of the quantity and measurement of signal strength.
        # Finally, the algorithm divides the mean-square value by the sample rate to normalize it to a single hertz. To obtain a linear value, we can take the square root of the mean-square value to obtain the root-mean-square (RMS).

        # save fft data              
        #np.save((pathlib.Path(path_data_preprocessing_subj) / '13a_snapshot_fftpowers'), fft_data_abs, allow_pickle=True)
        np.save((pathlib.Path(path_data_preprocessing_subj) / '13a_snapshot_amps'), amplitudes, allow_pickle=True)

        # compute powers from amplitudes
        powers = amplitudes ** 2

        # cycle through all epochs
        for _, epoch in enumerate(powers):  
            
            #cycle through all channels of epoch
            for channel_nr, channel in enumerate(epoch):

                for band in freq_bands:
                    fmin, fmax = freq_bands[band]

                    average_amp = np.mean(channel[(freqs >= fmin) & (freqs <= fmax)])

                    feature_name = f"{channel_name_list_eeg[channel_nr]}_{band}"

                    feature_dict[feature_name].append(average_amp)
        
        # build data frame
        feature_df = pd.DataFrame({key: pd.Series(val) for key, val in feature_dict.items() })

        # log 10 transform
        feature_df = np.log10(feature_df)      
        
        if save:
            feature_df.to_csv(path_data_preprocessing_subj / '13b_snapshot_features.csv')

### 14: Computing cross condition correlations

#### *crosscondition_correlations()*

Function that computes correlations between (features of) each epoch, then groups correlations according to the condition combination of the individual epocs and averages all correlations from each condition combination for each subject.

#### *pipeline_14_auto_crossconditioncorrelations()*

In [149]:
## Feature extraction
def crosscondition_correlations(feature_df, epochs, band = None, return_all_corrs = False):
    """
    Computes correlation matrices between epochs, grouped by condition, optionally for a specific frequency band.

    Args:
        feature_df (pd.DataFrame): DataFrame of features.
        epochs (mne.Epochs): Epochs object.
        band (str or None): Frequency band key.

    Returns:
        tuple: (correlation matrix, absolute correlation matrix)
    """

    if band is not None:
        band_key = f"_{band}"
        
        non_band_columns = []
        for i in range(len(feature_df.columns)):
            if band_key not in feature_df.columns[i]:
                non_band_columns.extend([feature_df.columns[i]])
        
        feature_df = feature_df.drop(columns = non_band_columns)
    
    epochs_correlation_matrix = feature_df.T.corr()
        
    # turn into long format   
    matrix_long = epochs_correlation_matrix.stack().reset_index()
    
    # delete all values from diagonal (where x and y index from the orginal dataframe are the same)
    matrix_long = matrix_long.drop(index = list(np.where(matrix_long.level_0 == matrix_long.level_1)[0])).reset_index().drop('index', axis = 1)
    
    # load epochs and get list of conditions 
    event_codes = [event[2] for event in epochs.events]
    
    # find condition group for each epoch index
    condition0, condition1 = [],[]
    for index in range(matrix_long.shape[0]):        
        condition0.extend([condition_dict[event_codes[matrix_long['level_0'][index]]]])
        condition1.extend([condition_dict[event_codes[matrix_long['level_1'][index]]]])  
    matrix_long['level_0'] = condition0
    matrix_long['level_1'] = condition1
    matrix_long.columns = ['level_0', 'level_1', 'corr']

    all_corrs = matrix_long.copy()

    # average correlation coefficients per condition group
    matrix_agg = matrix_long.copy().groupby(by = ['level_0', 'level_1'], sort = False).mean()

    # 
    matrix_long_abs = matrix_long.copy()
    # use only absolute correlation coefficients
    matrix_long_abs["corr"] = matrix_long_abs["corr"].abs()
    # average absolute correlation coefficients per condition group 
    matrix_agg_abs = matrix_long_abs.groupby(by = ['level_0', 'level_1'], sort = False).mean()

    # turn into wide format again to get correlation matrix
    cor_matrix = matrix_agg.reset_index().pivot(index='level_0', columns='level_1', values='corr')
    cor_matrix_abs  = matrix_agg_abs.reset_index().pivot(index='level_0', columns='level_1', values='corr')

    if return_all_corrs:
        all_corrs = matrix_long.copy()
        result = cor_matrix, cor_matrix_abs, all_corrs
    else:
        result = cor_matrix, cor_matrix_abs

    return(result)


def pipeline_14_auto_crossconditioncorrelations(subjects, save = True):
    """
    Computes and saves cross-condition correlation matrices for each subject.

    Args:
        subjects (list): List of subject identifiers.
        save (bool): Whether to save the result.
    """

    for subject in subjects:

        print(f"======================================================================================================================")
        print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Pipeline 14 - Auto - Computing cross condition correlations // Subject: {subject} ===============")

        set_current_subject(subject)
    
        log.append_line(f"Computing cross-condition correlations - {datetime.datetime.now().strftime('%H:%M:%S')}")
        
        # Load dataframe of features for each epoch
        feature_df = pd.read_csv(path_data_preprocessing_subj / '13b_snapshot_features.csv', index_col = 0)
        epochs = mne.read_epochs(path_data_preprocessing_subj / '12a_snapshot_epo.fif')

        # create and save subjectwise correlation matrix
        cor_matrix_all, cor_matrix_all_abs, all_corrs = crosscondition_correlations(feature_df, epochs, return_all_corrs = True)

        if save:
            cor_matrix_all.to_csv(path_data_preprocessing_subj / '14a_cond_corrmatrix.csv')
            cor_matrix_all_abs.to_csv(path_data_preprocessing_subj / '14b_cond_corrmatrix_abs.csv')
            all_corrs.to_csv(path_data_preprocessing_subj / '14c_all_correlations.csv')
        
        # create and save band wise correlation matrices
        for band in freq_bands.keys():
            band_corrmatrix, band_corrmatrix_abs = crosscondition_correlations(feature_df, epochs, band , return_all_corrs = False)

            if save:
                band_corrmatrix.to_csv(path_data_preprocessing_subj / f'14d_cond_corrmatrix_{band}.csv')
                band_corrmatrix_abs.to_csv(path_data_preprocessing_subj / f'14e_cond_corrmatrix_{band}_abs.csv')


### 15: Preprocessing behavioral data

load behavioral data for each subject from bids folder and save as csv in subject wise preprocessing folder

In [ ]:
def pipeline_15_preprocessing_behavioral_data(subjects, save = True):
    """
    Loads and processes behavioral data for each subject, saving as CSV.

    Args:
        subjects (list): List of subject identifiers.
        save (bool): Whether to save the result.
    """
    
    for subject in subjects:

        set_current_subject(subject)
    
        print(f"======================================================================================================================")
        print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Pipeline 15 - Auto - Preprocessing behavioral data // Subject: {subject} ===============")
        log.append_line(f"Preprocessing behav data - {datetime.datetime.now().strftime('%H:%M:%S')}")


        # subject behavioral data folder
        path_sourcedata_subj = path_sourcedata / f"sub-{str(subject).zfill(3)}"

        # list all files in folder
        all_files = os.listdir(path_sourcedata_subj)

        # identify ct logfile
        search_key = f"{str(subject).zfill(3)}_ct_log"
        subject_ct_logs = [file for file in all_files if search_key in file]

        if len(subject_ct_logs) > 1:
            raise Exception(f"More than one behavioral logfile found for subject: '{subject}'")
        else:
            subject_ct_log = subject_ct_logs[0]

        # load ct logfile
        subject_ct_log = path_sourcedata_subj / subject_ct_log
        subject_ct_df = pd.read_csv(subject_ct_log, encoding='cp1252')

        # add column for subject id and turn trial number into consequtive number
        subject_ct_df.insert(0, "subject", subject)
        subject_ct_df['trial'] = (subject_ct_df['block']-1)*15 + subject_ct_df['trial']

        # save data
        if save:
            subject_ct_df.to_csv(path_data_preprocessing_subj / '15a_ct_behav.csv', index=False)

        # identify rs logfile
        rs_search_key = f"{str(subject).zfill(3)}_rs_log"
        subject_rs_logfiles = [file for file in all_files if rs_search_key in file]

        if len(subject_rs_logfiles) > 1:
            raise Exception(f"More than one rs logfile found for subject: '{subject}'")
        else:
            subject_rs_log = subject_rs_logfiles[0]


        # load subject rs data
        subject_rs_log_file = path_sourcedata_subj / subject_rs_log
        subject_rs_df = pd.read_csv(subject_rs_log_file,encoding='cp1252')

        # add column for subject id
        subject_rs_df.insert(0, "subject", subject)

        # save data
        if save:
            subject_rs_df.to_csv(path_data_preprocessing_subj / '15b_rs_behav.csv', index=False)

### 16: Add trial info and ratings to table of epochs

#### *combine_ctrsepochs()*

#### *pipeline_16_combine_ctrsepochs()*

In [151]:
def pipeline_16_combine_ctrsepochs(subjects, save = True):
    """
    Combines behavioral and EEG epoch data, aligns trials, and saves combined features and ratings.

    Args:
        subjects (list): List of subject identifiers.
        save (bool): Whether to save the result.
    """
    
    for subject in subjects:

        print(f"======================================================================================================================")
        print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Pipeline 16 - Auto - Combine rs, ct and epoch data // Subject: {subject} ===============")

        set_current_subject(subject)

        # log.append_line(f"Combining rs, ct and epoch data - {datetime.datetime.now().strftime('%H:%M:%S')}")

        # ------------------------------------------------------------------------------------------------------
        # Combine data from rs and trial logfiles

        # load rs data
        subject_rs_df = pd.read_csv(path_data_preprocessing_subj / '15b_rs_behav.csv')
            
        # load ct data
        subject_ct_df = pd.read_csv(path_data_preprocessing_subj / '15a_ct_behav.csv')

        # compute duration
        duration_rs1 = subject_rs_df['end_time'][0] - subject_rs_df['starting_time'][0]
        duration_rs2 = subject_rs_df['end_time'][1] - subject_rs_df['starting_time'][1]

        # Add rs data to behav_df
        first_rs = [subject, int(0), 1, subject_rs_df['starting_time'][0], 226, duration_rs1, None, None, None, None, subject_rs_df['potj'][0], subject_rs_df['potj_rt'][0], None]  # adding a row
        second_rs = [subject, int(0), 2, subject_rs_df['starting_time'][1], 226, duration_rs2, None, None, None, None, subject_rs_df['potj'][1], subject_rs_df['potj_rt'][1], None]  # adding a row
        behav_df = subject_ct_df.copy()

        # add first rs as first line
        first_row_df = pd.DataFrame([first_rs], columns=behav_df.columns)
        behav_df = pd.concat([first_row_df, behav_df], ignore_index=True)

        # sort by index and reset index
        behav_df = behav_df.sort_index()
        behav_df = behav_df.reset_index(drop = True)

        # add sencond rs as last line
        last_row_df = pd.DataFrame([second_rs], columns=behav_df.columns)
        behav_df = pd.concat([behav_df, last_row_df], ignore_index=True)

        # set start_time of first trial (rs) as new baseline
        behav_df['starting_time'] = (behav_df['starting_time'] - behav_df['starting_time'][0])

        # change data types
        behav_df = behav_df.astype(datatypes_dict)


        # ------------------------------------------------------------------------------------------------------
        # Check whether sequence of conditions in logfiles matches sequence of conditions in eeg markers

        # get codes from eeg events (epochs are already filtered with some events missing)
        clean_ica_eeg = mne.io.read_raw_fif(path_data_preprocessing_subj / '11a_lp_nobad_hp_ica_reref_clean_eeg.fif', preload = True)
        
        # extract events
        eeg_events, _ = mne.events_from_annotations(clean_ica_eeg, event_id = event_dict)

        # extract list of task events codes from eeg logfile (based on triggers)
        marker_events = [eeg_event for eeg_event in eeg_events if eeg_event[2] in analysis_ids]

        # translate markers to readable condition names
        eeg_codes = [[key for key, value in event_dict.items() if value == eeg_task_event[2]][0] for eeg_task_event in marker_events]

        # extract list of task event codes from behavioral logfile (based on logfile entries from presentation pc) and translate to readable condition names
        behav_codes = [logfile_marker_translation[str(trial)] for trial in behav_df["condition_code"]]

        # check if both lists correspond
        eeg_codes == behav_codes

        # Check if number of markers in raw data correspond to expected number of rs (2) and task events (75)
        if len(marker_events) != 77:
            rs_events = [event for event in marker_events if eeg_event[2] in rs_ids]
            if len(rs_events) != 2:
                raise Exception (f"Unexpected number of RS events: {len(rs_events)}")

            task_events = [event for event in marker_events if eeg_event[2] in task_ids]
            if len(task_events) != 75:
                raise Exception (f"Unexpected number of task events: {len(task_events)}")


        # ------------------------------------------------------------------------------------------------------
        # use time stamps from marker list as starting point in behav file

        # get time stamps from marker list
        marker_starts = [event[0] for event in marker_events]

        marker_start_sec = [(start / resample_freq) for start in marker_starts]

        # use time stamps as starting point in behav file
        behav_df["starting_time"] = marker_start_sec


        # ------------------------------------------------------------------------------------------------------
        # Retrieve snapshot events and convert starting time to secs 

        # load epochs
        epochs = mne.read_epochs(path_data_preprocessing_subj / '12a_snapshot_epo.fif', preload = True)

        # delete events not relevant for analysis
        snapshot_events = [np.array(event) for event in epochs.events if event[2] in analysis_ids]

        # compute new starting time in seconds
        snapshot_events = [[(event[0]/resample_freq), event[1], event[2]] for event in snapshot_events]

        # find the matching trial for each epoch
        trial_indices = []
        within_trial_indices = []
        epoch_starts = []

        for _, event in enumerate(snapshot_events):

            # get start of the event
            event_start = event[0]
            epoch_starts.append(event_start)

            # find highest index of trials with starting_time <= event_start 
            last_lower = np.max([index for index, row in behav_df.iterrows() if (row["starting_time"] <= event_start)])

            # extract condition of current event according to identified trial
            current_trial_code = logfile_marker_translation[str(behav_df['condition_code'][last_lower])]

            # update trial code
            current_trial_code = logfile_marker_translation[str(behav_df['condition_code'][last_lower])]

            # ---------------------------------------
            # finally check wether codes match and raise exception if not

            # extract condition of current event according to markers
            event_code = [key for key, value in event_dict.items() if value == event[2]][0]

            if current_trial_code != event_code:
                raise Exception (f"Codes do not match: event_code = {event_code} / current_trial_code: {current_trial_code}")   
                
            # add new trial index
            trial_indices.extend([last_lower])
            
            # find the number of the snapshot within the trial
            snapshot_nr = (event_start - behav_df["starting_time"][last_lower]) // snapshot_epochs_duration
            within_trial_indices.append(round(snapshot_nr))


        # ------------------------------------------------------------------------------------------------------
        # Load feature df

        # load epoch features
        feature_df_original = pd.read_csv(path_data_preprocessing_subj / '13b_snapshot_features.csv', index_col = 0)

        # list all events not relevant to analysis
        non_analysis_events = [event_nr for event_nr, event in enumerate(epochs.events) if not event[2] in analysis_ids]

        # drop entries in feature_df of all events not relevant to analysis
        feature_df = feature_df_original.copy().drop(index = non_analysis_events).reset_index()


        # ------------------------------------------------------------------------------------------------------
        # Combine trial information and features for each epoch

        # Create a new df for combined epoch and trial information
        features_and_ratings = feature_df.copy()
        features_and_ratings.insert(0, "epoch_start", epoch_starts)
        features_and_ratings.insert(0, "snapshot_nr", within_trial_indices)
        features_and_ratings.insert(0, "trial_nr", trial_indices)

        # retrieve conditions for each each epoch based on trigger codes
        con_list, vel_list, den_list, rating_list = [],[],[],[]
        for _, row in features_and_ratings.iterrows():
            con_list.extend([logfile_marker_translation[str(behav_df['condition_code'][row["trial_nr"]])]])
            vel_list.extend([behav_df['velocity'][row["trial_nr"]]])
            den_list.extend([behav_df['density'][row["trial_nr"]]])
            rating_list.extend([behav_df['time_rating'][row["trial_nr"]]])

        # add condition information to combined epoch / trial info list
        features_and_ratings.insert(0, "subject", subject)
        features_and_ratings.insert(1, "condition", con_list)
        features_and_ratings.insert(2, "velocity", vel_list)
        features_and_ratings.insert(3, "density", den_list)
        features_and_ratings.insert(4, "potj", rating_list)

        # save combined epoch / trial info list
        if save:
            features_and_ratings.to_csv(path_data_preprocessing_subj / "16_features_and_behav.csv", index = False)

# Running the Pipeline

define first (start) and last (end) subject for batch processing

In [152]:
# Misc
save = True
overwrite = False
#3
use_old_bads = True
#5
use_old_annotations_ica = True
#6
use_old_ica = True
#7
use_old_ica_components = True
#9
run_without_plotting = True
#11
use_old_annotations = True
use_ica_annotations = False
show_data = True


# Select subjects for processing / empty list means all subjects will be processed
subject_selection = []

# select subjects from specific group for processing
single_group = "controls" # "patients" # "controls" # "all" # False

# DO NOT CHANGE FOR SUBJECTWISE PREPROCESSING (Preprocessed demodata file containing information about inclusion of subjects is created in groupwise preprocessing script and not available yet)
exclude_subjects = False
use_raw_demodata = True


# Retrieve subject list based on specified paramters
subjects = get_subject_list(use_raw_demodata = use_raw_demodata, exclude_subjects = exclude_subjects, single_group = single_group, subject_selection = subject_selection)

"""
pipeline_1_auto_preparedataset(subjects, save = save, overwrite = overwrite) #cropping, adding annotations, downsampling

pipeline_2_auto_lowpassfiltering(subjects, save = save, overwrite = overwrite)

pipeline_3_manual_badchannelexclusion(subjects, use_old_bads = use_old_bads, save = save, overwrite = overwrite)

pipeline_4_auto_highpassfiltering(subjects, save = save, overwrite = overwrite)

pipeline_5_manual_badsegmentexclusion(subjects, use_old_annotations = use_old_annotations_ica, save = save, overwrite = overwrite)

pipeline_6_auto_fittingica(subjects, use_old_ica = use_old_ica, save = save, overwrite = overwrite)

pipeline_7_manual_identifyingicacomponents(subjects, use_old_ica_components = use_old_ica_components, save = save, overwrite = overwrite)

pipeline_8_auto_apllyica(subjects, save = save, overwrite = overwrite)

pipeline_9_manual_comparebeforeafterica(subjects, run_without_plotting = run_without_plotting)

pipeline_10_auto_interpolateandrereference(subjects, save = save, overwrite = overwrite)

pipeline_11_manual_excludingartefacts(subjects, use_old_annotations = use_old_annotations, use_ica_annotations = use_ica_annotations, show_data = show_data, save = save, overwrite = overwrite) 

pipeline_12_auto_epoching(subjects, save = save, overwrite = overwrite)

pipeline_13_auto_featureextraction(subjects, save = save)

pipeline_14_auto_crossconditioncorrelations(subjects, save = save)

"""

pipeline_15_preprocessing_behavioral_data(subjects, save = save)

pipeline_16_combine_ctrsepochs(subjects, save = save)


print(f"======================================================================================================================")
print(f"=============== {datetime.datetime.now().strftime('%H:%M:%S')} // Pipeline finished ===============================================")
print(f"======================================================================================================================")

logfile path: D:\EEGST data test\preprocessing\001\preprocessing_logfile_001.txt
=============== 12:53:56 // Pipeline 15 - Auto - Preprocessing behavioral data // Subject: 1 ===============
logfile path: D:\EEGST data test\preprocessing\002\preprocessing_logfile_002.txt
=============== 12:53:56 // Pipeline 15 - Auto - Preprocessing behavioral data // Subject: 2 ===============
logfile path: D:\EEGST data test\preprocessing\003\preprocessing_logfile_003.txt
=============== 12:53:56 // Pipeline 15 - Auto - Preprocessing behavioral data // Subject: 3 ===============
=============== 12:53:56 // Pipeline 16 - Auto - Combine rs, ct and epoch data // Subject: 1 ===============
logfile path: D:\EEGST data test\preprocessing\001\preprocessing_logfile_001.txt
Opening raw data file D:\EEGST data test\preprocessing\001\11a_lp_nobad_hp_ica_reref_clean_eeg.fif...


C:\Users\jordingma\AppData\Local\Temp\ipykernel_7156\1950397727.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  behav_df = pd.concat([first_row_df, behav_df], ignore_index=True)
C:\Users\jordingma\AppData\Local\Temp\ipykernel_7156\1950397727.py:47: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  behav_df = pd.concat([behav_df, last_row_df], ignore_index=True)


    Range : 2207 ... 790888 =      8.828 ...  3163.552 secs
Ready.
Reading 0 ... 788681  =      0.000 ...  3154.724 secs...
Used Annotations descriptions: ['block_end', 'block_start', 'colorchange', 'rating_color', 'rating_time', 'rs_end', 'rs_rating_end', 'rs_rating_start', 'rs_start', 'starfield/cc/vel0/den1', 'starfield/cc/vel0/den2', 'starfield/cc/vel1/den1', 'starfield/cc/vel1/den2', 'starfield/cc/vel2/den1', 'starfield/cc/vel2/den2', 'starfield/nc/vel0/den1', 'starfield/nc/vel0/den2', 'starfield/nc/vel1/den1', 'starfield/nc/vel1/den2', 'starfield/nc/vel2/den1', 'starfield/nc/vel2/den2', 'trial_end']
Reading D:\EEGST data test\preprocessing\001\12a_snapshot_epo.fif ...
    Found the data of interest:
        t =    -200.00 ...    2000.00 ms
        0 CTF compensation matrices available
Not setting metadata
952 matching events found
No baseline correction applied
0 projection items activated
=============== 12:54:05 // Pipeline 16 - Auto - Combine rs, ct and epoch data // Subject: 

C:\Users\jordingma\AppData\Local\Temp\ipykernel_7156\1950397727.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  behav_df = pd.concat([first_row_df, behav_df], ignore_index=True)
C:\Users\jordingma\AppData\Local\Temp\ipykernel_7156\1950397727.py:47: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  behav_df = pd.concat([behav_df, last_row_df], ignore_index=True)


    Range : 5076 ... 820160 =     20.304 ...  3280.640 secs
Ready.
Reading 0 ... 815084  =      0.000 ...  3260.336 secs...
Used Annotations descriptions: ['block_end', 'block_start', 'colorchange', 'rating_color', 'rating_time', 'rs_end', 'rs_rating_end', 'rs_rating_start', 'rs_start', 'starfield/cc/vel0/den1', 'starfield/cc/vel0/den2', 'starfield/cc/vel1/den1', 'starfield/cc/vel1/den2', 'starfield/cc/vel2/den1', 'starfield/cc/vel2/den2', 'starfield/nc/vel0/den1', 'starfield/nc/vel0/den2', 'starfield/nc/vel1/den1', 'starfield/nc/vel1/den2', 'starfield/nc/vel2/den1', 'starfield/nc/vel2/den2', 'trial_end']
Reading D:\EEGST data test\preprocessing\002\12a_snapshot_epo.fif ...
    Found the data of interest:
        t =    -200.00 ...    2000.00 ms
        0 CTF compensation matrices available
Not setting metadata
926 matching events found
No baseline correction applied
0 projection items activated
=============== 12:54:12 // Pipeline 16 - Auto - Combine rs, ct and epoch data // Subject: 

C:\Users\jordingma\AppData\Local\Temp\ipykernel_7156\1950397727.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  behav_df = pd.concat([first_row_df, behav_df], ignore_index=True)
C:\Users\jordingma\AppData\Local\Temp\ipykernel_7156\1950397727.py:47: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  behav_df = pd.concat([behav_df, last_row_df], ignore_index=True)


    Range : 71339 ... 900480 =    285.356 ...  3601.920 secs
Ready.
Reading 0 ... 829141  =      0.000 ...  3316.564 secs...
Used Annotations descriptions: ['block_end', 'block_start', 'colorchange', 'rating_color', 'rating_time', 'rs_end', 'rs_rating_end', 'rs_rating_start', 'rs_start', 'starfield/cc/vel0/den1', 'starfield/cc/vel0/den2', 'starfield/cc/vel1/den2', 'starfield/cc/vel2/den1', 'starfield/cc/vel2/den2', 'starfield/nc/vel0/den1', 'starfield/nc/vel0/den2', 'starfield/nc/vel1/den1', 'starfield/nc/vel1/den2', 'starfield/nc/vel2/den1', 'starfield/nc/vel2/den2', 'trial_end']
Reading D:\EEGST data test\preprocessing\003\12a_snapshot_epo.fif ...
    Found the data of interest:
        t =    -200.00 ...    2000.00 ms
        0 CTF compensation matrices available
Not setting metadata
747 matching events found
No baseline correction applied
0 projection items activated
=============== 12:54:18 // Pipeline finished ===============================================
